# Introduction: Why Linear Algebra for Machine Learning

## What's covered

- Why linear algebra is the most important math foundation for machine learning
- The roadmap through this repo — 8 notebooks from linear equations to PCA
- Linear equations — one variable, two, three, and *n*; lines, planes, hyperplanes
- Systems of linear equations and the famous form `Ax = b`
- The three possible outcomes — unique solution, none, or infinitely many

Before we dive into vectors, matrices, and eigenvalues, this notebook does two things: motivate **why** linear algebra is the math foundation underneath every ML model, and warm up with **linear equations** — the simplest object in the subject — to see where they hide inside ML.


## Why machine learning cannot escape linear algebra

Every ML model, from logistic regression to a transformer, reduces to chains of operations on vectors and matrices. The short version, in six points:

**1. Data is a matrix.** A dataset with `n` samples and `d` features is an `n × d` matrix. Each row is one sample, each column is one feature. The moment you write `X.shape == (1000, 20)`, you are already working with a matrix.

**2. A single sample is a vector.** One row of `X` — say, one customer's features — is a `d`-dimensional vector. The whole point of *embeddings* is to represent things (words, images, users) as vectors so we can do math on them.

**3. A linear model is a matrix-vector product.** Linear regression is literally `y_hat = X @ w + b`. Logistic regression adds a sigmoid on top. A fully-connected neural network layer is `h = activation(W @ x + b)`. The neural net is just a stack of these.

**4. Training is solving systems and optimizing in vector spaces.** The closed-form fit for linear regression solves the linear system `(X^T X) w = X^T y`. Gradient descent walks through a vector space following the gradient — itself another vector.

**5. Distance and similarity are inner products and norms.** Cosine similarity between embeddings is `(u · v) / (||u|| ||v||)`. k-NN, recommendation systems, contrastive learning — all built on this.

**6. Dimensionality reduction is eigendecomposition.** PCA finds the directions of maximum variance in the data; those directions are eigenvectors of the covariance matrix. SVD, the workhorse behind PCA and recommender systems, is a matrix decomposition.

If you skip linear algebra, ML becomes a black box of `model.fit()` calls. If you internalize it, every model becomes a transparent composition of pieces you already understand.


## The roadmap

We will build up from the simplest objects to the ones you meet inside real ML papers:

1. **This notebook** — linear equations, the seed everything grows from
2. **Vectors** — dot products, norms, projections
3. **Vector spaces** — span, basis, linear independence; the geometry of "where data can live"
4. **Matrices** — linear transformations, multiplication, inverse, determinant
5. **Linear systems** — `Ax = b` in full generality, rank, the four fundamental subspaces
6. **Orthogonality** — projections, Gram-Schmidt, least-squares regression
7. **Eigendecomposition** — eigenvalues, eigenvectors, diagonalization
8. **SVD & PCA** — the most useful decomposition in machine learning

By the end you should be able to read a neural-network forward pass, or a PCA call, and see exactly what is happening underneath.


## What is a linear equation?

A **linear equation** is an equation where every variable appears to the power of 1, and variables are never multiplied by each other. That is the entire definition.

**One variable**

$$ ax = b $$

A single unknown `x`, given `a` and `b`. Solution: `x = b / a` (when `a ≠ 0`).

**Two variables**

$$ a_1 x + a_2 y = b $$

Geometrically, a **line** in 2D. Every point `(x, y)` on the line satisfies the equation.

**Three variables**

$$ a_1 x + a_2 y + a_3 z = b $$

Geometrically, a **plane** in 3D.

**n variables**

$$ a_1 x_1 + a_2 x_2 + \dots + a_n x_n = b $$

A **hyperplane** in n-dimensional space. You cannot picture it directly, but the algebra works exactly the same.

The key word is *linear*: no `x^2`, no `xy`, no `sin(x)`. If a model can be written in this form, we call it a *linear model* — and we get to use every tool in this repo.


In [ ]:
import numpy as np

# Solve 3x = 12
a, b = 3, 12
x = b / a
print(f"x = {x}")


## Systems of linear equations

In ML you rarely have a single equation. You have many, all to be satisfied at once.

**Example.** Two equations, two unknowns:

$$
\begin{aligned}
2x + y &= 5 \\
x - y &= 1
\end{aligned}
$$

Geometrically: two lines in the `(x, y)` plane. The solution is the single point where they **intersect**.

We can write this much more compactly using a matrix and two vectors:

$$
\underbrace{\begin{bmatrix} 2 & 1 \\ 1 & -1 \end{bmatrix}}_{A}
\underbrace{\begin{bmatrix} x \\ y \end{bmatrix}}_{\mathbf{x}}
=
\underbrace{\begin{bmatrix} 5 \\ 1 \end{bmatrix}}_{\mathbf{b}}
$$

This is the famous form **`Ax = b`** — the central object of all of linear algebra. Notebook 5 is dedicated entirely to it.


In [ ]:
A = np.array([[2, 1],
              [1, -1]])
b = np.array([5, 1])

x = np.linalg.solve(A, b)
print(f"x = {x[0]}, y = {x[1]}")

# Verify
print(f"A @ x = {A @ x}")
print(f"target b = {b}")


## Three cases: unique, none, or infinite

For a system `Ax = b` there are exactly three possible outcomes. The geometric picture is worth more than memorizing the algebra.

**Case 1 — Unique solution.** The lines (or planes, or hyperplanes) intersect at a single point. The happy path; `np.linalg.solve` returns the answer.

**Case 2 — No solution.** The lines are parallel and distinct. The system is *inconsistent*. In ML this happens all the time: with more equations than unknowns and noisy data, you cannot satisfy everything exactly. We then look for the *best approximate* solution — that is what **least squares** does, covered in the orthogonality notebook.

**Case 3 — Infinitely many solutions.** The two equations describe the *same* line. The system is *underdetermined*. In ML this shows up when you have more features than samples; the model has freedom in how it sets weights. **Regularization** (ridge, lasso) is how we pick one specific solution out of infinitely many.


In [ ]:
# Case 1: unique solution
A1 = np.array([[2, 1], [1, -1]])
b1 = np.array([5, 1])
print("Case 1 (unique):  ", np.linalg.solve(A1, b1))

# Case 2: no solution — parallel lines, x + y = 2 and x + y = 5
A2 = np.array([[1, 1], [1, 1]])
b2 = np.array([2, 5])
try:
    sol = np.linalg.solve(A2, b2)
    print("Case 2 (inconsistent):", sol)
except np.linalg.LinAlgError as e:
    print(f"Case 2 (inconsistent): LinAlgError — {e}")

# Case 3: infinite solutions — same line, x + y = 2 and 2x + 2y = 4
A3 = np.array([[1, 1], [2, 2]])
b3 = np.array([2, 4])
try:
    sol = np.linalg.solve(A3, b3)
    print("Case 3 (infinite):   ", sol)
except np.linalg.LinAlgError as e:
    print(f"Case 3 (infinite): LinAlgError — {e}")


## Where this appears in ML

The humble `Ax = b` is hiding inside almost every model you will work with:

- **Linear regression (closed form).** Solving the *normal equations* `(X^T X) w = X^T y` for the weights `w` — one big linear system.
- **Ridge regression.** Solving `(X^T X + λI) w = X^T y`. Adding `λI` guarantees a unique solution even when `X^T X` is rank-deficient — that is Case 3 above, fixed by regularization.
- **PCA.** Eigenvectors of the covariance matrix are solutions to `(A - λI) v = 0`, a special linear system.
- **Backpropagation.** Each layer's gradient computation is a matrix-vector product, applied recursively through the network.
- **Recommender systems.** Matrix factorization solves systems to recover missing entries in a user-item matrix.

You are already partway to understanding all of these. Next notebook: **vectors**.
